In [7]:
import requests
import pandas as pd
import os

In [8]:
fire_event_name = "YORK_2024-08-03_5"
job_id = '573421be-412c-403d-9d72-b591de886c43'
test = requests.get(f"https://fire-recovery-backend-dev-113009620257.us-central1.run.app/fire-recovery/result/analyze_fire_severity/{fire_event_name}/{job_id}")
print(test.json())

{'fire_event_name': 'YORK_2024-08-03_5', 'status': 'pending', 'job_id': '573421be-412c-403d-9d72-b591de886c43'}


In [ ]:
fires = pd.read_csv('fire_processing_jobs.csv')

# Collect status for each row
statuses = []

for idx, row in fires.iterrows():
    # Skip rows with missing fire_event_name or job_id
    if pd.isna(row['fire_event_name']) or pd.isna(row['job_id']):
        statuses.append(999)
        continue
        
    if row['status'] == 'success':
        fire_event_name = row['fire_event_name']
        job_id = row['job_id']

        request = requests.get(f"https://fire-recovery-backend-dev-113009620257.us-central1.run.app/fire-recovery/result/analyze_fire_severity/{fire_event_name}/{job_id}")
        
        status = request.json().get('status')
        print(f"Job {fire_event_name}: {status}")
        
    else:
        status = 999
    
    statuses.append(status)

# Add status column to dataframe
fires['job_status'] = statuses

# Save updated dataframe
fires.to_csv('fire_processing_jobs.csv', index=False)

print(f"\nUpdated fire_processing_jobs.csv with job_status column")
fires.head()

Job COFFEE POT_2024-08-03_5: complete
Job COFFEE POT_2024-08-03_10: complete
Job COFFEE POT_2024-08-03_15: complete
Job COFFEE POT_2024-08-03_21: complete
Job COFFEE POT_2024-08-03_30: complete
Job COFFEE POT_2024-08-03_45: complete
Job COFFEE POT_2024-08-03_60: complete
Job COFFEE POT_2024-08-03_90: complete
Job SENTINEL_2024-07-14_5: complete
Job SENTINEL_2024-07-14_10: complete


In [ ]:
urls = request.json().get('coarse_severity_cog_urls')
os.makedirs('data/raw', exist_ok=True)

# Download the three files
for metric in ['rbr', 'dnbr', 'rdnbr']:
    response = requests.get(urls[metric])
    with open(f"data/raw/{fire_event_name}_{metric}.tif", 'wb') as f:
        f.write(response.content)

print(f"Job {fire_event_name} downloaded.")
return True